# 02 - Exploratory Data Analysis (Credit Card Transactions)

**Phase goal:** Understand the structure, distributions, relationships, and fraud patterns in the cleaned transaction data before any feature engineering or modelling.

## Analysis sections

| # | Section | Key Questions |
|---|---------|---------------|
| 1 | Setup & Data Loading | Load Parquet, confirm schema and row counts |
| 2 | Dataset Overview | Shape, dtypes, null counts, descriptive stats |
| 3 | Target Variable Analysis | Class balance, fraud rate, imbalance implications |
| 4 | Transaction Amount Analysis | Distribution shape, skew, outliers, fraud vs legit |
| 5 | Temporal Patterns | Hour-of-day, day-of-week, monthly trends |
| 6 | Categorical Analysis | Category, merchant, state fraud rates |
| 7 | Geographic Analysis | Cardholder vs merchant location, distance |
| 8 | Demographic Analysis | Age, gender spending and fraud patterns |
| 9 | Correlation Analysis | Pearson heatmap, pairplot on numeric features |
| 10 | Fraud Signal Summary | Ranked fraud indicators for downstream phases |

**Input:** `data/processed/transactions_parquet` (from notebook 01)  
**Output:** EDA figures saved to `notebooks/eda_figures/`

## 1. Setup & Data Loading

In [ ]:
# Install dependencies if needed (Colab)
# !pip install -r requirements.txt

from pathlib import Path
import sys
import math

cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline import create_spark_session
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Stop any existing Spark session before creating a new one
from pyspark.sql import SparkSession
existing = SparkSession.getActiveSession()
if existing:
    existing.stop()

processed_dir = project_root / "data" / "processed"
input_parquet  = processed_dir / "transactions_parquet"
figures_dir    = project_root / "notebooks" / "eda_figures"
figures_dir.mkdir(parents=True, exist_ok=True)

spark = create_spark_session(app_name="CreditCardEDA")
print(f"Input  : {input_parquet}")
print(f"Figures: {figures_dir}")

In [ ]:
df = spark.read.parquet(str(input_parquet))
print(f"Loaded {df.count():,} rows | {len(df.columns)} columns")
df.printSchema()
df.show(5, truncate=False)

## 2. Dataset Overview

Check shape, null counts, and basic descriptive statistics across all numeric columns.

In [ ]:
print(f"Shape: {df.count():,} rows  x  {len(df.columns)} columns")

print("\n=== Null Counts ===")
null_counts = df.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_pct"] = (null_counts["null_count"] / df.count() * 100).round(2)
print(null_counts[null_counts["null_count"] > 0].to_string())

print("\n=== Descriptive Statistics (numeric columns) ===")
numeric_cols = [c for c in ["amount", "is_fraud", "lat", "long",
                             "city_pop", "unix_time", "merch_lat", "merch_long"]
                if c in df.columns]
df.select(numeric_cols).describe().show(truncate=False)

### 2. Dataset Overview — Findings

The dataset contains **1,296,675 transactions across 23 columns** after cleaning in notebook 01. 
Only one column has missing values: `merch_zipcode` with 195,973 nulls (15.11%) — all other 
columns are complete. Core numeric columns show the following key statistics:

- **amount:** mean $70.35, std $160.32, max $28,948.90 — heavily right-skewed, most transactions are small
- **is_fraud:** mean 0.0058 — confirms fraud is a rare event (<1% of transactions)
- **lat/long vs merch_lat/merch_long:** nearly identical means and ranges, both centered around the continental US (38°N, -90°W)
- **unix_time:** spans January 2012 to June 2013 — approximately 18 months of transaction history
- **city_pop:** ranges from 23 to 2,906,700 — cardholders live in both rural and major urban areas

## 3. Target Variable Analysis

Examine the class distribution of the `is_fraud` label.

In [ ]:
total   = df.count()
n_fraud = df.filter(F.col("is_fraud") == 1).count()
n_legit = df.filter(F.col("is_fraud") == 0).count()

print("=== Class Distribution ===")
print(f"{'label':>12} {'count':>10} {'pct':>8}")
print(f"{'Legitimate':>12} {n_legit:>10,} {100*n_legit/total:>7.3f}")
print(f"{'Fraud':>12} {n_fraud:>10,} {100*n_fraud/total:>7.3f}")
print(f"\nOverall fraud rate: {100*n_fraud/total:.3f}%")
print(f"Imbalance ratio  : {n_legit/n_fraud:.1f}:1 (legit:fraud)")

# Plot
labels = ["Legitimate", "Fraud"]
counts = [n_legit, n_fraud]
pcts   = [100*n_legit/total, 100*n_fraud/total]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Target Variable  Class Balance", fontsize=14, fontweight="bold")

bars = axes[0].bar(labels, counts, color=["steelblue", "tomato"], alpha=0.85)
for bar, cnt, pct in zip(bars, counts, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f"{cnt:,}\n({pct:.2f}%)", ha="center", va="bottom", fontsize=10)
axes[0].set_ylabel("Transaction Count")
axes[0].set_title("Class Distribution: Legitimate vs Fraud")

axes[1].pie(counts, labels=labels, colors=["steelblue", "tomato"],
            autopct="%1.3f%%", startangle=90)
axes[1].set_title("Class Distribution (Proportional)")

plt.tight_layout()
out = figures_dir / "target_distribution.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

### 3. Target Variable Analysis — Findings

The dataset contains **1,296,675 transactions** split into **1,289,169 legitimate (99.42%)** 
and **7,506 fraud (0.58%)**, yielding a severe class imbalance ratio of **171.8:1**.

A naive model predicting "legitimate" for every transaction would achieve 99.42% accuracy 
while catching zero fraud — making accuracy a meaningless metric here. This imbalance 
must be addressed before any modelling through resampling techniques, and evaluation 
must rely on AUC-ROC, Precision, Recall and F1 rather than accuracy.

## 4. Transaction Amount Analysis

Examine the distribution of transaction amounts, skewness, outliers, and how amounts differ between legitimate and fraudulent transactions.

In [ ]:
print("=== Amount Statistics by Fraud Label ===")
df.groupBy("is_fraud").agg(
    F.round(F.mean("amount"), 2).alias("mean"),
    F.round(F.percentile_approx("amount", 0.5), 2).alias("median"),
    F.round(F.stddev("amount"), 2).alias("std"),
    F.round(F.min("amount"), 2).alias("min"),
    F.round(F.percentile_approx("amount", 0.25), 2).alias("p25"),
    F.round(F.percentile_approx("amount", 0.75), 2).alias("p75"),
    F.round(F.percentile_approx("amount", 0.95), 2).alias("p95"),
    F.round(F.max("amount"), 2).alias("max")
).orderBy("is_fraud").show()

In [ ]:
# Sample for plotting
sample_pdf = df.select("amount", "is_fraud").sample(0.1, seed=42).toPandas()
sample_pdf["amount_log1p"] = np.log1p(sample_pdf["amount"])

legit = sample_pdf[sample_pdf["is_fraud"] == 0]
fraud = sample_pdf[sample_pdf["is_fraud"] == 1]

# p99 clip for boxplot
p99 = sample_pdf["amount"].quantile(0.99)
clipped = sample_pdf[sample_pdf["amount"] <= p99].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Transaction Amount Analysis", fontsize=14, fontweight="bold")

# Raw distribution
axes[0,0].hist(legit["amount"], bins=80, density=True, alpha=0.6, color="steelblue", label="Legitimate")
axes[0,0].hist(fraud["amount"], bins=80, density=True, alpha=0.6, color="tomato", label="Fraud")
axes[0,0].set_xlabel("Transaction Amount ($)")
axes[0,0].set_ylabel("Density")
axes[0,0].set_title("Amount Distribution: Legitimate vs Fraud")
axes[0,0].legend()

# log1p distribution
axes[0,1].hist(legit["amount_log1p"], bins=60, density=True, alpha=0.6, color="steelblue", label="Legitimate")
axes[0,1].hist(fraud["amount_log1p"], bins=60, density=True, alpha=0.6, color="tomato", label="Fraud")
axes[0,1].set_xlabel("log1p(Amount)")
axes[0,1].set_ylabel("Density")
axes[0,1].set_title("log1p(Amount) Distribution")
axes[0,1].legend()

# Boxplot clipped at p99
fraud_label = clipped["is_fraud"].map({0: "Legitimate", 1: "Fraud"})
clipped_copy = clipped.copy()
clipped_copy["label"] = fraud_label
clipped_copy.boxplot(column="amount", by="label", ax=axes[1,0],
                     boxprops=dict(color="steelblue"),
                     medianprops=dict(color="navy"))
axes[1,0].set_title("Amount Box Plot (clipped at p99)")
axes[1,0].set_xlabel("")
axes[1,0].set_ylabel("Transaction Amount ($)")
plt.sca(axes[1,0])
plt.title("Amount Box Plot (clipped at p99)")

# CDF
for label, color, name in [(0, "steelblue", "Legitimate"), (1, "tomato", "Fraud")]:
    subset = sample_pdf[sample_pdf["is_fraud"] == label]["amount"].sort_values()
    cdf = np.arange(1, len(subset)+1) / len(subset)
    axes[1,1].plot(subset.values[:int(len(subset)*0.99)],
                   cdf[:int(len(subset)*0.99)],
                   color=color, label=name)
axes[1,1].set_xlabel("Transaction Amount ($)")
axes[1,1].set_ylabel("Cumulative Probability")
axes[1,1].set_title("Empirical CDF: Amount")
axes[1,1].legend()

plt.tight_layout()
out = figures_dir / "amount_analysis.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

### 4. Transaction Amount Analysis — Findings

Fraud transactions are on average **~8x larger** than legitimate ones ($531 vs $68 mean), 
with even the 25th percentile of fraud ($245) exceeding the 75th percentile of legitimate 
transactions ($82) — the two distributions occupy almost completely different ranges.

The raw distribution is heavily right-skewed — most legitimate transactions cluster under 
$100 while a small number reach $28,948 — meaning the data will need transformation before 
modelling to compress this extreme range. The log1p plot makes the class separation far 
more visible, the boxplot shows non-overlapping interquartile ranges between classes, and 
the CDF confirms that 90% of legitimate transactions fall under $100 while fraud is spread 
broadly up to $1,376. Transaction amount is the strongest raw fraud signal in the dataset.

## 5. Temporal Patterns

Analyse fraud rates and transaction volumes by hour of day, day of week, and month.

In [ ]:
# Parse transaction timestamp
ts_col = "trans_date_trans_time" if "trans_date_trans_time" in df.columns else "transaction_ts"
df = df.withColumn("tx_ts",         F.to_timestamp(F.col(ts_col)))
df = df.withColumn("tx_hour",       F.hour("tx_ts"))
df = df.withColumn("tx_day_of_week",F.dayofweek("tx_ts"))
df = df.withColumn("tx_month_str",  F.date_format("tx_ts", "yyyy-MM"))

# Hourly
hour_pdf = (
    df.groupBy("tx_hour")
      .agg(F.count("*").alias("tx_count"),
           (F.mean("is_fraud")*100).alias("fraud_rate_pct"))
      .orderBy("tx_hour")
      .toPandas()
)

print("=== Hourly Summary (top 5 fraud hours) ===")
print(hour_pdf.nlargest(5, "fraud_rate_pct")[["tx_hour","tx_count","fraud_rate_pct"]].to_string(index=False))

# Day of week
dow_pdf = (
    df.groupBy("tx_day_of_week")
      .agg(F.count("*").alias("tx_count"),
           (F.mean("is_fraud")*100).alias("fraud_rate_pct"))
      .orderBy("tx_day_of_week")
      .toPandas()
)
day_labels = ["Sun","Mon","Tue","Wed","Thu","Fri","Sat"]
dow_pdf["day_label"] = dow_pdf["tx_day_of_week"].apply(lambda x: day_labels[x-1] if 1<=x<=7 else str(x))

# Monthly
month_pdf = (
    df.groupBy("tx_month_str")
      .agg(F.count("*").alias("tx_count"),
           (F.mean("is_fraud")*100).alias("fraud_rate_pct"))
      .orderBy("tx_month_str")
      .toPandas()
)

# Plot
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle("Temporal Patterns", fontsize=14, fontweight="bold")

axes[0,0].bar(hour_pdf["tx_hour"], hour_pdf["tx_count"], color="steelblue", alpha=0.85)
axes[0,0].set_xlabel("Hour of Day"); axes[0,0].set_ylabel("Transaction Count")
axes[0,0].set_title("Transaction Volume by Hour"); axes[0,0].set_xticks(range(24))

axes[0,1].plot(hour_pdf["tx_hour"], hour_pdf["fraud_rate_pct"], color="tomato", marker="o", linewidth=2)
axes[0,1].fill_between(hour_pdf["tx_hour"], hour_pdf["fraud_rate_pct"], alpha=0.2, color="tomato")
axes[0,1].set_xlabel("Hour of Day"); axes[0,1].set_ylabel("Fraud Rate (%)")
axes[0,1].set_title("Fraud Rate by Hour of Day"); axes[0,1].set_xticks(range(24))

axes[1,0].bar(dow_pdf["day_label"], dow_pdf["tx_count"], color="orange", alpha=0.85)
axes[1,0].set_xlabel("Day of Week"); axes[1,0].set_ylabel("Transaction Count")
axes[1,0].set_title("Transaction Volume by Day of Week")

bars = axes[1,1].bar(dow_pdf["day_label"], dow_pdf["fraud_rate_pct"], color="tomato", alpha=0.85)
for bar, val in zip(bars, dow_pdf["fraud_rate_pct"]):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                   f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
axes[1,1].set_xlabel("Day of Week"); axes[1,1].set_ylabel("Fraud Rate (%)")
axes[1,1].set_title("Fraud Rate by Day of Week")

axes[2,0].bar(month_pdf["tx_month_str"], month_pdf["tx_count"], color="steelblue", alpha=0.85)
axes[2,0].set_xlabel("Month"); axes[2,0].set_ylabel("Transaction Count")
axes[2,0].set_title("Monthly Transaction Volume")
plt.setp(axes[2,0].xaxis.get_majorticklabels(), rotation=45, ha="right")

axes[2,1].plot(month_pdf["tx_month_str"], month_pdf["fraud_rate_pct"], color="tomato", marker="o", linewidth=2)
axes[2,1].set_xlabel("Month"); axes[2,1].set_ylabel("Fraud Rate (%)")
axes[2,1].set_title("Monthly Fraud Rate")
plt.setp(axes[2,1].xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.tight_layout()
out = figures_dir / "temporal_patterns.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

### 5. Temporal Patterns — Findings

Fraud peaks sharply at **10-11 PM** (2.88-2.84%) — nearly 5x the dataset average of 
0.58% — then drops to near zero during daytime hours (6AM-8PM ~0.1%). Despite lower 
transaction volume in the early morning hours, fraud rate remains elevated through 2AM, 
suggesting fraudsters deliberately target hours when cardholders are asleep and less 
likely to notice suspicious charges.

By day of week, fraud is lowest on Sunday/Monday (0.47-0.48%) and highest 
Thursday-Friday (0.68-0.71%). Monthly fraud rate shows no strong seasonal trend despite 
a volume spike in November-December 2019 during the holiday season. Hour of day is the 
strongest temporal fraud signal by far — a useful feature for downstream modelling.

## 6. Categorical Analysis

Examine fraud rates across transaction categories, top merchants, and states.

In [ ]:
print("=== Category Breakdown ===")
cat_pdf = (
    df.groupBy("category")
      .agg(F.count("*").alias("tx_count"),
           F.round(F.mean("amount"), 2).alias("avg_amount"),
           F.round(F.mean("is_fraud")*100, 3).alias("fraud_rate_pct"),
           F.round(F.sum("amount"), 2).alias("total_spend"))
      .orderBy(F.col("tx_count").desc())
      .toPandas()
)
print(cat_pdf.to_string(index=False))

# Top merchants by fraud rate (min 50 transactions)
merch_pdf = (
    df.groupBy("merchant")
      .agg(F.count("*").alias("tx_count"),
           F.round(F.mean("is_fraud")*100, 3).alias("fraud_rate_pct"))
      .filter(F.col("tx_count") >= 50)
      .orderBy(F.col("fraud_rate_pct").desc())
      .limit(10)
      .toPandas()
)

# State breakdown
state_pdf = (
    df.groupBy("state")
      .agg(F.count("*").alias("tx_count"),
           F.round(F.mean("is_fraud")*100, 3).alias("fraud_rate_pct"))
      .orderBy(F.col("tx_count").desc())
      .limit(15)
      .toPandas()
)

overall_avg = df.select(F.mean("is_fraud")*100).first()[0]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Categorical Analysis", fontsize=14, fontweight="bold")

# Volume by category
cat_sorted = cat_pdf.sort_values("tx_count", ascending=True)
axes[0,0].barh(cat_sorted["category"], cat_sorted["tx_count"], color="steelblue", alpha=0.85)
axes[0,0].set_xlabel("Transaction Count"); axes[0,0].set_title("Transaction Count by Category")

# Fraud rate by category
cat_fraud = cat_pdf.sort_values("fraud_rate_pct", ascending=True)
axes[0,1].barh(cat_fraud["category"], cat_fraud["fraud_rate_pct"], color="tomato", alpha=0.85)
axes[0,1].axvline(overall_avg, color="black", linestyle="--", label="Overall avg")
axes[0,1].set_xlabel("Fraud Rate (%)"); axes[0,1].set_title("Fraud Rate by Category")
axes[0,1].legend()

# Top merchants by fraud rate
axes[1,0].barh(merch_pdf["merchant"], merch_pdf["fraud_rate_pct"], color="tomato", alpha=0.85)
axes[1,0].set_xlabel("Fraud Rate (%)"); axes[1,0].set_title("Top 10 Merchants by Fraud Rate (min 50 tx)")

# State volume + fraud rate
ax2 = axes[1,1].twinx()
axes[1,1].bar(state_pdf["state"], state_pdf["tx_count"], color="steelblue", alpha=0.6, label="Volume")
ax2.plot(state_pdf["state"], state_pdf["fraud_rate_pct"], color="tomato", marker="o", label="Fraud Rate %")
axes[1,1].set_xlabel("State"); axes[1,1].set_ylabel("Transaction Count")
ax2.set_ylabel("Fraud Rate (%)")
axes[1,1].set_title("Top 15 States: Volume & Fraud Rate")
plt.setp(axes[1,1].xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.tight_layout()
out = figures_dir / "categorical_analysis.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

### 6. Categorical Analysis — Findings

Online categories carry the highest fraud risk — `shopping_net` (1.76%), `misc_net` 
(1.45%), and `grocery_pos` (1.41%) all exceed the 0.58% average, while in-person 
services like `health_fitness` (0.16%) and `home` (0.16%) are the safest — an 11x 
difference between extremes. The highest volume category (`gas_transport`) has a 
below-average fraud rate (0.47%), confirming that volume alone does not drive fraud risk.

At the merchant level, top fraudulent merchants reach 2.5% fraud rate — more than 4x 
the dataset average — indicating that individual merchant identity carries significant 
predictive value. State-level fraud rates vary independently of transaction volume, 
suggesting geography at the state level is not a strong standalone predictor.

## 7. Geographic Analysis

Examine the relationship between cardholder location, merchant location, and fraud.

In [ ]:
from pyspark.sql.types import DoubleType

@F.udf(returnType=DoubleType())
def haversine_km(lat1, lon1, lat2, lon2):
    if any(v is None for v in [lat1, lon1, lat2, lon2]):
        return None
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

geo_cols = {"lat", "long", "merch_lat", "merch_long"}
if geo_cols.issubset(set(df.columns)):
    df = df.withColumn("dist_km",
        haversine_km(F.col("lat"), F.col("long"), F.col("merch_lat"), F.col("merch_long")))

    # Distance bands
    df = df.withColumn("dist_bin",
        F.when(F.col("dist_km") < 10,  "< 10 km")
         .when(F.col("dist_km") < 50,  "10-50 km")
         .when(F.col("dist_km") < 100, "50-100 km")
         .otherwise("100-300 km"))

    dist_pdf = (
        df.groupBy("dist_bin")
          .agg(F.count("*").alias("tx_count"),
               F.round(F.mean("is_fraud")*100, 3).alias("fraud_rate_pct"))
          .toPandas()
    )
    print("=== Fraud Rate by Distance Band ===")
    print(dist_pdf.to_string(index=False))

    # Sample for plotting
    geo_pdf = df.select("lat", "long", "dist_km", "is_fraud").dropna().sample(0.05, seed=42).toPandas()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Geographic Analysis", fontsize=14, fontweight="bold")

    # Fraud rate by distance band
    band_order = ["< 10 km", "10-50 km", "50-100 km", "100-300 km"]
    dist_ordered = dist_pdf.set_index("dist_bin").reindex(band_order).reset_index()
    bars = axes[0].bar(dist_ordered["dist_bin"], dist_ordered["fraud_rate_pct"], color="tomato", alpha=0.85)
    for bar, val in zip(bars, dist_ordered["fraud_rate_pct"]):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                     f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
    axes[0].set_xlabel("Cardholder-Merchant Distance"); axes[0].set_ylabel("Fraud Rate (%)")
    axes[0].set_title("Fraud Rate by Distance Band")

    # Distance distribution
    legit_geo = geo_pdf[geo_pdf["is_fraud"]==0]["dist_km"]
    fraud_geo = geo_pdf[geo_pdf["is_fraud"]==1]["dist_km"]
    axes[1].hist(legit_geo, bins=60, density=True, alpha=0.6, color="steelblue", label="Legitimate")
    axes[1].hist(fraud_geo, bins=60, density=True, alpha=0.6, color="tomato", label="Fraud")
    axes[1].set_xlabel("Distance (km)"); axes[1].set_ylabel("Density")
    axes[1].set_title("Distance Distribution: Legit vs Fraud")
    axes[1].legend()

    # Cardholder map
    legit_map = geo_pdf[geo_pdf["is_fraud"]==0]
    fraud_map = geo_pdf[geo_pdf["is_fraud"]==1]
    axes[2].scatter(legit_map["long"], legit_map["lat"], s=1, alpha=0.3, color="steelblue", label="Legitimate")
    axes[2].scatter(fraud_map["long"], fraud_map["lat"], s=8, alpha=0.8, color="tomato", label="Fraud")
    axes[2].set_xlabel("Longitude"); axes[2].set_ylabel("Latitude")
    axes[2].set_title("Cardholder Locations (US)")
    axes[2].legend(markerscale=5)

    plt.tight_layout()
    out = figures_dir / "geographic_analysis.png"
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"Saved: {out}")
else:
    print("Skipping geo analysis: lat/long columns not found.")

### 7. Geographic Analysis — Findings

Cardholder-merchant distance shows **minimal variation** in fraud rate across all distance 
bands — ranging only from 0.50% (<10km) to 0.59% (50-100km), all close to the 0.58% 
dataset average. The distance distribution plot confirms legitimate and fraud transactions 
follow nearly identical distance patterns, peaking around 60-100km for both classes.

This suggests that in this dataset, fraudsters operate at similar distances to legitimate 
cardholders — likely because most fraud is card-not-present (online) where physical 
distance is irrelevant. The cardholder map confirms transactions are spread across the 
continental US with no obvious geographic clustering of fraud. Distance between cardholder 
and merchant is likely a weak predictor in this dataset.

## 8. Demographic Analysis

Examine spending and fraud patterns by cardholder age group and gender.

In [ ]:
if "dob" in df.columns:
    df = df.withColumn("dob_ts", F.to_date(F.col("dob"), "yyyy-MM-dd")) \
           .withColumn("age",
               F.floor(F.datediff(F.col("tx_ts").cast("date"), F.col("dob_ts")) / 365.25))

    df = df.withColumn("age_group",
        F.when(F.col("age") < 25, "Under 25")
         .when(F.col("age") < 35, "25-34")
         .when(F.col("age") < 45, "35-44")
         .when(F.col("age") < 55, "45-54")
         .when(F.col("age") < 65, "55-64")
         .otherwise("65+")
    )

age_pdf = (
    df.groupBy("age_group")
      .agg(F.count("*").alias("tx_count"),
           F.round(F.mean("amount"), 2).alias("avg_amount"),
           F.round(F.mean("is_fraud")*100, 3).alias("fraud_rate_pct"))
      .toPandas()
)
age_order = ["Under 25","25-34","35-44","45-54","55-64","65+"]
age_pdf = age_pdf.set_index("age_group").reindex(age_order).reset_index()

gender_pdf = (
    df.groupBy("gender")
      .agg(F.count("*").alias("tx_count"),
           F.round(F.mean("amount"), 2).alias("avg_amount"),
           F.round(F.mean("is_fraud")*100, 3).alias("fraud_rate_pct"),
           F.round(F.sum("amount"), 2).alias("total_spend"))
      .orderBy("gender")
      .toPandas()
)

print("=== Gender Breakdown ===")
print(gender_pdf.to_string(index=False))
print("\n=== Age Group Breakdown ===")
print(age_pdf.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Demographic Analysis", fontsize=14, fontweight="bold")

axes[0,0].bar(age_pdf["age_group"], age_pdf["avg_amount"], color="steelblue", alpha=0.85)
axes[0,0].set_xlabel("Age Group"); axes[0,0].set_ylabel("Avg Transaction Amount ($)")
axes[0,0].set_title("Avg Amount by Age Group")

bars = axes[0,1].bar(age_pdf["age_group"], age_pdf["fraud_rate_pct"], color="tomato", alpha=0.85)
for bar, val in zip(bars, age_pdf["fraud_rate_pct"]):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                   f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
axes[0,1].set_xlabel("Age Group"); axes[0,1].set_ylabel("Fraud Rate (%)")
axes[0,1].set_title("Fraud Rate by Age Group")

axes[1,0].bar(gender_pdf["gender"], gender_pdf["avg_amount"], color=["steelblue","tomato"], alpha=0.85)
axes[1,0].set_xlabel("Gender"); axes[1,0].set_ylabel("Avg Transaction Amount ($)")
axes[1,0].set_title("Avg Amount by Gender")

bars = axes[1,1].bar(gender_pdf["gender"], gender_pdf["fraud_rate_pct"], color=["steelblue","tomato"], alpha=0.85)
for bar, val in zip(bars, gender_pdf["fraud_rate_pct"]):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                   f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
axes[1,1].set_xlabel("Gender"); axes[1,1].set_ylabel("Fraud Rate (%)")
axes[1,1].set_title("Fraud Rate by Gender")

plt.tight_layout()
out = figures_dir / "demographic_analysis.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

### 8. Demographic Analysis — Findings

**Age:** Fraud risk follows a U-shaped pattern — highest for the youngest (Under 25: 0.63%) 
and oldest (55-64: 0.77%, 65+: 0.74%) cardholders, lowest for middle-aged (35-44: 0.43%). 
Older cardholders are likely targeted more due to less familiarity with fraud detection, 
while younger cardholders may engage in riskier online spending. The nearly 2x difference 
between safest (35-44: 0.43%) and riskiest (55-64: 0.77%) age groups suggests cardholder 
age is a useful feature for downstream modelling.

**Gender:** Males have a slightly higher fraud rate (0.64%) vs females (0.53%) with 
virtually identical average transaction amounts (~$70). The difference is modest — only 
0.11 percentage points — making gender a weak but non-zero signal on its own.

## 9. Correlation Analysis

Compute Pearson correlations between numeric features and the fraud label, and visualise pairwise relationships.

In [ ]:
corr_cols = [c for c in ["amount", "city_pop", "is_fraud", "age",
                          "dist_km", "tx_hour", "tx_month_str", "lat"]
             if c in df.columns]

# Use tx_month as numeric
df = df.withColumn("tx_month_num", F.month("tx_ts"))

corr_cols = [c for c in ["amount", "city_pop", "is_fraud", "age",
                          "dist_km", "tx_hour", "tx_month_num"]
             if c in df.columns]

# Sample for correlation (Spark ML Correlation needs Vector)
corr_sample = df.select(corr_cols).dropna().sample(0.05, seed=42)
print(f"Correlation sample size: {corr_sample.count():,}")

assembler = VectorAssembler(inputCols=corr_cols, outputCol="features", handleInvalid="skip")
corr_input = assembler.transform(corr_sample).select("features")
corr_matrix = Correlation.corr(corr_input, "features", method="pearson").head()[0]
corr_array  = corr_matrix.toArray()
corr_pdf = pd.DataFrame(corr_array, index=corr_cols, columns=corr_cols)

print("\n=== Correlations with is_fraud (descending) ===")
if "is_fraud" in corr_pdf.columns:
    print(corr_pdf["is_fraud"].sort_values(key=abs, ascending=False).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Correlation Analysis", fontsize=14, fontweight="bold")

sns.heatmap(corr_pdf, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=axes[0], square=True)
axes[0].set_title("Pearson Correlation Heatmap")

if "is_fraud" in corr_pdf.columns:
    fraud_corr = corr_pdf["is_fraud"].drop("is_fraud").sort_values()
    colors = ["tomato" if v > 0 else "steelblue" for v in fraud_corr]
    axes[1].barh(fraud_corr.index, fraud_corr.values, color=colors, alpha=0.85)
    axes[1].axvline(0, color="black", linewidth=0.8)
    axes[1].set_xlabel("Pearson Correlation with is_fraud")
    axes[1].set_title("Feature Correlations with Fraud Label")

plt.tight_layout()
out = figures_dir / "correlation_analysis.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

In [ ]:
# Pairplot on a small sample
pair_cols = [c for c in ["amount", "tx_hour", "age", "dist_km", "is_fraud"] if c in df.columns]
pair_pdf = df.select(pair_cols).dropna().sample(0.03, seed=42).toPandas()
pair_pdf["amount_log1p"] = np.log1p(pair_pdf["amount"])

plot_cols = [c for c in ["amount_log1p", "tx_hour", "age", "dist_km"] if c in pair_pdf.columns]

g = sns.pairplot(pair_pdf, vars=plot_cols, hue="is_fraud",
                 palette={0: "steelblue", 1: "tomato"},
                 plot_kws={"alpha": 0.3, "s": 5},
                 diag_kind="kde")
g.fig.suptitle("Pairwise Feature Relationships", y=1.02, fontsize=13)

out = figures_dir / "pairplot.png"
g.fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

### 9. Correlation Analysis — Findings

**Correlations with is_fraud (Pearson):**

| Feature | Correlation | Strength |
|---|---|---|
| amount | 0.188 | Moderate — strongest predictor |
| age | 0.013 | Very weak positive |
| tx_hour | 0.010 | Very weak positive |
| city_pop | 0.007 | Negligible |
| tx_month | -0.008 | Negligible negative |
| dist_km | -0.004 | Negligible |

Amount dominates all other features at 0.188 — nearly 15x stronger than the next 
feature. The pairplot confirms this visually — amount (log-transformed) is the only 
feature where fraud points visibly separate from legitimate. All other features show 
fraud and legitimate points completely overlapping, meaning their relationship with 
fraud is non-linear rather than absent — something a linear correlation cannot capture 
but tree-based models handle well in downstream modelling.

## 10. Fraud Signal Summary

Ranked summary of all fraud signals discovered in this EDA, ordered by predictive strength.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    EDA — FRAUD SIGNAL SUMMARY                        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  STRONG SIGNALS                                                      ║
║  • Transaction amount  — fraud txns ~8x larger (Pearson 0.188)      ║
║  • Merchant category   — shopping_net 1.76% vs health 0.16% (11x)  ║
║  • Merchant identity   — top merchants reach 2.5% fraud rate (4x)  ║
║                                                                      ║
║  MODERATE SIGNALS                                                    ║
║  • Hour of day         — 10-11 PM peaks at 2.88% (5x avg)          ║
║  • Cardholder age      — 55-64 group at 0.77% vs 35-44 at 0.43%    ║
║                                                                      ║
║  WEAK SIGNALS                                                        ║
║  • Gender              — M 0.64% vs F 0.53% (minimal difference)    ║
║  • Day of week         — Thu-Fri slightly higher (0.68-0.71%)       ║
║  • Geographic distance — no meaningful variation across bands        ║
║  • City population     — Pearson 0.007, negligible                  ║
║                                                                      ║
║  KEY MODELLING IMPLICATIONS                                          ║
║  • Class imbalance 171.8:1 — resampling required before training    ║
║  • Accuracy is meaningless — use AUC-ROC, F1, Precision, Recall     ║
║  • Most signals are non-linear — tree-based models preferred        ║
║  • Amount needs log transformation to compress extreme skew         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

spark.stop()
print("Spark session stopped.")

### 10. Fraud Signal Summary — Findings

Transaction amount is the single strongest fraud signal, with fraud transactions averaging 
8x more than legitimate ones and a Pearson correlation of 0.188. Merchant category and 
individual merchant identity are the next strongest signals, with an 11x fraud rate 
difference between the safest and riskiest categories. Hour of day and cardholder age 
provide moderate additional signal. Geographic distance, gender, and city population 
are weak predictors on their own.

The severe class imbalance (171.8:1) and predominantly non-linear relationships between 
features and fraud label mean that downstream modelling must address resampling, avoid 
accuracy as a metric, and favour tree-based models that capture non-linear patterns. 
These findings directly inform the feature engineering and modelling decisions in 
notebooks 03 and 04.